# Training Checkpoint Progression Analysis

Plot how key evaluation metrics evolve across training checkpoints.

**Input format**: each entry in `CHECKPOINT_ROOT_PATHS` must be a checkpoint root directory
containing a `benchmark_export/{subset}/` subdirectory (e.g., `.../checkpoint-950/benchmark_export/valid/`).

Optionally, `BASE_MODEL_ROOT_PATH` can point to a directory with the same structure holding
evaluation results for the **base model** (before any training). When set, each plot draws
horizontal dashed reference lines (or a distinct marker) at the base model's metric values.

In [ ]:
import dataclasses
import pathlib
import re
import typing

import matplotlib.axes
import matplotlib.figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pyine.data.utils.lmdb_io
import pyine.evals.code_exec.analysis
import pyine.evals.code_exec.reeval
import pyine.evals.code_exec.utils
import pyine.utils.filesystem

In [ ]:
# --- configuration ---

CHECKPOINT_ROOT_PATHS: list[str] = [
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-50",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-100",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-150",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-200",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-250",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-300",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-350",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-400",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-450",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-500",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-550",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-600",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-650",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-700",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-750",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-800",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-850",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-900",
    "../data/RL_HT_49_evals/evals/v0_rl_eval_base_RL_HT_49_checkpoint-950",
]

# explicit step numbers; if None, auto-extracted from `checkpoint-{N}` in path components
CHECKPOINT_STEPS: list[int] | None = None

# optional path to a base model evaluation directory (same structure as checkpoint dirs);
# when set, plots include horizontal dashed reference lines at the base model's metric values
BASE_MODEL_ROOT_PATH: str | None = "../data/RL_HT_49_evals/evals/v0_rl_eval_base_Qwen3-4B-Instruct-2507"

TARGET_EVAL_SUBSET_NAME: typing.Literal["valid", "test"] = "valid"

# --- plot settings ---
FIGSIZE_SINGLE: tuple[int, int] = (10, 6)
FIGSIZE_MULTI: tuple[int, int] = (14, 5)
SHOW_CI_BANDS: bool = True
CI_BAND_ALPHA: float = 0.15
BASE_MODEL_LINE_ALPHA: float = 0.55
BASE_MODEL_LINE_STYLE: str = "--"

In [ ]:
MATCH_TYPES = pyine.evals.code_exec.utils.MATCH_TYPES
MATCH_TYPE_LABELS = pyine.evals.code_exec.analysis.MATCH_TYPE_LABELS
MetricWithCI = pyine.evals.code_exec.analysis.MetricWithCI


@dataclasses.dataclass
class CheckpointRecord:
    """Holds all loaded data for a single training checkpoint."""

    step: int
    raw_path: pathlib.Path
    lmdb_path: pathlib.Path
    eval_result: pyine.evals.code_exec.utils.CodeExecEvalResult
    summary: pyine.evals.code_exec.analysis.EvalRunSummary
    is_base_model: bool = False


def extract_checkpoint_step(
    path: pathlib.Path,
) -> int | None:
    """Extract step number from a path component matching ``checkpoint-(\\d+)``."""
    pattern = re.compile(r"checkpoint-(\d+)")
    for part in reversed(path.parts):
        match = pattern.search(part)
        if match:
            return int(match.group(1))
    return None

In [ ]:
# --- path resolution, loading & validation ---

if not CHECKPOINT_ROOT_PATHS:
    raise ValueError("CHECKPOINT_ROOT_PATHS must be non-empty")
if CHECKPOINT_STEPS is not None and len(CHECKPOINT_STEPS) != len(CHECKPOINT_ROOT_PATHS):
    raise ValueError(
        f"CHECKPOINT_STEPS length ({len(CHECKPOINT_STEPS)}) must match "
        f"CHECKPOINT_ROOT_PATHS length ({len(CHECKPOINT_ROOT_PATHS)})"
    )

records: list[CheckpointRecord] = []
for path_idx, root_str in enumerate(CHECKPOINT_ROOT_PATHS):
    root_path = pathlib.Path(root_str)
    lmdb_path = root_path / "benchmark_export" / TARGET_EVAL_SUBSET_NAME
    if not (lmdb_path / "data.mdb").exists():
        raise FileNotFoundError(
            f"No LMDB found at {lmdb_path} (expected data.mdb); "
            f"ensure the checkpoint has a benchmark_export/{TARGET_EVAL_SUBSET_NAME}/ subdirectory"
        )
    # resolve step number
    if CHECKPOINT_STEPS is not None:
        step = CHECKPOINT_STEPS[path_idx]
    else:
        step = extract_checkpoint_step(root_path)
        if step is None:
            raise ValueError(f"Cannot auto-extract step from path {root_path}; set CHECKPOINT_STEPS explicitly")
    # load eval result from LMDB
    eval_result = pyine.evals.code_exec.reeval.reconstruct_from_lmdb(
        [lmdb_path],
        eval_subset_name=TARGET_EVAL_SUBSET_NAME,
    )
    summary = pyine.evals.code_exec.analysis.eval_result_to_summary(
        eval_result,
        subset_name=TARGET_EVAL_SUBSET_NAME,
        run_name=f"step-{step}",
        source_path=lmdb_path,
    )
    records.append(
        CheckpointRecord(
            step=step,
            raw_path=root_path,
            lmdb_path=lmdb_path,
            eval_result=eval_result,
            summary=summary,
        )
    )
    print(
        f"  [{path_idx + 1}/{len(CHECKPOINT_ROOT_PATHS)}] step={step}: "
        f"{eval_result.num_samples} samples, {eval_result.num_attempts} attempts"
    )

# --- hard validation ---
all_steps = [r.step for r in records]
if len(set(all_steps)) != len(all_steps):
    raise ValueError(f"Duplicate checkpoint steps detected: {all_steps}")

# validate identical sample identifiers across all checkpoints
reference_ids = set(records[0].eval_result.unique_sample_identifiers)
for record in records[1:]:
    current_ids = set(record.eval_result.unique_sample_identifiers)
    if current_ids != reference_ids:
        only_in_ref = reference_ids - current_ids
        only_in_cur = current_ids - reference_ids
        raise ValueError(
            f"Sample identity mismatch at step {record.step}: "
            f"{len(only_in_ref)} samples only in step {records[0].step}, "
            f"{len(only_in_cur)} samples only in step {record.step}"
        )

# validate identical attempt counts
reference_attempts = records[0].eval_result.num_attempts
for record in records[1:]:
    if record.eval_result.num_attempts != reference_attempts:
        raise ValueError(
            f"Attempt count mismatch: step {records[0].step} has {reference_attempts}, "
            f"step {record.step} has {record.eval_result.num_attempts}"
        )

# validate identical pass@k key sets
reference_pass_at_k_keys = set(records[0].summary.run_info.pass_at_k.keys())
for record in records[1:]:
    current_keys = set(record.summary.run_info.pass_at_k.keys())
    if current_keys != reference_pass_at_k_keys:
        raise ValueError(
            f"Pass@K key mismatch: step {records[0].step} has {reference_pass_at_k_keys}, "
            f"step {record.step} has {current_keys}"
        )

# sort by step
records.sort(key=lambda r: r.step)
print(f"\nLoaded {len(records)} checkpoints (steps: {[r.step for r in records]})")
print(f"  samples per checkpoint: {records[0].eval_result.num_samples}")
print(f"  attempts per checkpoint: {records[0].eval_result.num_attempts}")

# --- optional base model loading ---
base_record: CheckpointRecord | None = None
if BASE_MODEL_ROOT_PATH is not None:
    base_path = pathlib.Path(BASE_MODEL_ROOT_PATH)
    base_lmdb = base_path / "benchmark_export" / TARGET_EVAL_SUBSET_NAME
    if not (base_lmdb / "data.mdb").exists():
        raise FileNotFoundError(
            f"No LMDB found at {base_lmdb} (expected data.mdb); "
            f"ensure the base model dir has a benchmark_export/{TARGET_EVAL_SUBSET_NAME}/ subdirectory"
        )
    base_eval_result = pyine.evals.code_exec.reeval.reconstruct_from_lmdb(
        [base_lmdb],
        eval_subset_name=TARGET_EVAL_SUBSET_NAME,
    )
    base_summary = pyine.evals.code_exec.analysis.eval_result_to_summary(
        base_eval_result,
        subset_name=TARGET_EVAL_SUBSET_NAME,
        run_name="base",
        source_path=base_lmdb,
    )
    base_record = CheckpointRecord(
        step=0,
        raw_path=base_path,
        lmdb_path=base_lmdb,
        eval_result=base_eval_result,
        summary=base_summary,
        is_base_model=True,
    )
    print(f"\n  [base model]: {base_eval_result.num_samples} samples, {base_eval_result.num_attempts} attempts")
    # warn (don't error) if base model has different sample/attempt counts
    if base_eval_result.num_samples != records[0].eval_result.num_samples:
        print(
            f"  WARNING: base model has {base_eval_result.num_samples} samples "
            f"vs {records[0].eval_result.num_samples} in checkpoints"
        )
    if base_eval_result.num_attempts != records[0].eval_result.num_attempts:
        print(
            f"  WARNING: base model has {base_eval_result.num_attempts} attempts "
            f"vs {records[0].eval_result.num_attempts} in checkpoints"
        )

In [ ]:
# --- summary DataFrame ---
all_summaries = ([base_record.summary] if base_record is not None else []) + [r.summary for r in records]
summary_df = pyine.evals.code_exec.analysis.summarize_runs_to_dataframe(all_summaries)
all_step_labels = (["base"] if base_record is not None else []) + [r.step for r in records]
summary_df.insert(0, "checkpoint_step", all_step_labels)
summary_df  # noqa: B018

In [ ]:
# --- plotting helper functions ---


def _draw_base_hline(
    ax: matplotlib.axes.Axes,
    value: float | None,
    color: typing.Any,
    label: str | None = None,
) -> None:
    """Draw a horizontal dashed reference line for a base model metric value."""
    if value is None or np.isnan(value):
        return
    ax.axhline(
        value,
        color=color,
        linestyle=BASE_MODEL_LINE_STYLE,
        alpha=BASE_MODEL_LINE_ALPHA,
        linewidth=1.5,
        label=label,
    )


def plot_training_curves(
    records: list[CheckpointRecord],
    base_record: CheckpointRecord | None = None,
    figsize: tuple[int, int] = FIGSIZE_SINGLE,
    show_ci: bool = SHOW_CI_BANDS,
    ci_alpha: float = CI_BAND_ALPHA,
) -> matplotlib.figure.Figure:
    """Plot overall accuracy over training steps, one line per match type."""
    steps = [r.step for r in records]
    tab10_colors = plt.cm.tab10.colors  # type: ignore[reportAttributeAccessIssue]
    fig, ax = plt.subplots(figsize=figsize)
    for mt_idx, match_type in enumerate(MATCH_TYPES):
        values = [r.summary.run_info.accuracy.get(match_type) for r in records]
        if all(v is None for v in values):
            continue
        y_vals = [v.value if v is not None else np.nan for v in values]
        color = tab10_colors[mt_idx % len(tab10_colors)]
        label = MATCH_TYPE_LABELS[match_type]
        ax.plot(steps, y_vals, "o-", color=color, label=label, linewidth=2, markersize=5)
        if show_ci:
            ci_lo = [v.ci_lower if v is not None and v.ci_lower is not None else np.nan for v in values]
            ci_hi = [v.ci_upper if v is not None and v.ci_upper is not None else np.nan for v in values]
            if not all(np.isnan(ci_lo)) and not all(np.isnan(ci_hi)):
                ax.fill_between(steps, ci_lo, ci_hi, color=color, alpha=ci_alpha)
        if base_record is not None:
            base_metric = base_record.summary.run_info.accuracy.get(match_type)
            base_val = base_metric.value if base_metric is not None else None
            _draw_base_hline(ax, base_val, color, label=f"{label} (base)")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"Overall Accuracy over Training ({TARGET_EVAL_SUBSET_NAME} set)")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    return fig


def _compute_category_jitter_offsets(
    steps: list[int],
    num_categories: int,
) -> np.ndarray:
    """Compute per-category x-offsets so error bars don't stack on top of each other."""
    if num_categories <= 1 or len(steps) <= 1:
        return np.zeros(max(num_categories, 1))
    step_gap = min(steps[idx + 1] - steps[idx] for idx in range(len(steps) - 1))
    jitter_total = step_gap * 0.25  # spread across 25% of the smallest gap
    return np.linspace(-jitter_total / 2, jitter_total / 2, num_categories)


def plot_category_training_curves(
    records: list[CheckpointRecord],
    category_prefix: str,
    match_type: pyine.evals.code_exec.utils.MatchType,
    ax: matplotlib.axes.Axes | None = None,
    base_record: CheckpointRecord | None = None,
    figsize: tuple[int, int] = FIGSIZE_SINGLE,
    show_ci: bool = SHOW_CI_BANDS,
    ci_capsize: float = 2,
    ci_capthick: float = 0.8,
    ci_linewidth: float = 0.8,
    ci_alpha: float = 0.45,
) -> matplotlib.figure.Figure:
    """Plot per-category accuracy over training steps for a given match type.

    CI is shown as thin error bars with slight x-jitter per category (cleaner than
    overlapping fill_between bands when many categories share the same y-range).
    """
    # discover all unique categories matching prefix across all checkpoints
    all_categories: list[str] = []
    for record in records:
        for cat_metric in record.summary.category_metrics:
            if cat_metric.category.startswith(category_prefix) and cat_metric.category not in all_categories:
                all_categories.append(cat_metric.category)
    all_categories.sort()
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = typing.cast("matplotlib.figure.Figure", ax.get_figure())
    steps = [r.step for r in records]
    tab10_colors = plt.cm.tab10.colors  # type: ignore[reportAttributeAccessIssue]
    jitter_offsets = _compute_category_jitter_offsets(steps, len(all_categories))
    # build a lookup: step -> category -> CategoryMetrics for fast access
    cat_lookup: dict[int, dict[str, pyine.evals.code_exec.analysis.CategoryMetrics]] = {}
    for record in records:
        cat_lookup[record.step] = {
            cm.category: cm for cm in record.summary.category_metrics if cm.category.startswith(category_prefix)
        }
    # build base model category lookup
    base_cat_map: dict[str, pyine.evals.code_exec.analysis.CategoryMetrics] = {}
    if base_record is not None:
        base_cat_map = {
            cm.category: cm for cm in base_record.summary.category_metrics if cm.category.startswith(category_prefix)
        }
    warnings: list[str] = []
    for cat_idx, category in enumerate(all_categories):
        short_label = category.removeprefix(category_prefix)
        color = tab10_colors[cat_idx % len(tab10_colors)]
        y_vals: list[float] = []
        ci_lo_vals: list[float] = []
        ci_hi_vals: list[float] = []
        sample_counts: list[int] = []
        for record in records:
            cat_metric = cat_lookup[record.step].get(category)
            if cat_metric is None:
                y_vals.append(np.nan)
                ci_lo_vals.append(np.nan)
                ci_hi_vals.append(np.nan)
                sample_counts.append(0)
            else:
                metric_ci = cat_metric.accuracy.get(match_type, MetricWithCI())
                y_vals.append(metric_ci.value if metric_ci.value is not None else np.nan)
                ci_lo_vals.append(metric_ci.ci_lower if metric_ci.ci_lower is not None else np.nan)
                ci_hi_vals.append(metric_ci.ci_upper if metric_ci.ci_upper is not None else np.nan)
                sample_counts.append(cat_metric.sample_count)
        # annotate sample count from last checkpoint; flag drift
        last_count = sample_counts[-1] if sample_counts else 0
        has_drift = len({sc for sc in sample_counts if sc > 0}) > 1
        count_suffix = f" (n={last_count}{'*' if has_drift else ''})"
        if has_drift:
            warnings.append(f"  {short_label}: sample counts vary across checkpoints: {sample_counts}")
        # jittered x positions for this category
        x_jittered = [s + jitter_offsets[cat_idx] for s in steps]
        ax.plot(x_jittered, y_vals, "o-", color=color, label=short_label + count_suffix, linewidth=1.5, markersize=4)
        if show_ci:
            y_arr = np.array(y_vals)
            ci_lo_arr = np.array(ci_lo_vals)
            ci_hi_arr = np.array(ci_hi_vals)
            # only draw error bars at points where both CI bounds are available
            valid_mask = ~(np.isnan(ci_lo_arr) | np.isnan(ci_hi_arr) | np.isnan(y_arr))
            if valid_mask.any():
                x_eb = np.array(x_jittered)[valid_mask]
                y_eb = y_arr[valid_mask]
                yerr_lo = y_eb - ci_lo_arr[valid_mask]
                yerr_hi = ci_hi_arr[valid_mask] - y_eb
                ax.errorbar(
                    x_eb,
                    y_eb,
                    yerr=[yerr_lo, yerr_hi],
                    fmt="none",
                    color=color,
                    alpha=ci_alpha,
                    capsize=ci_capsize,
                    capthick=ci_capthick,
                    linewidth=ci_linewidth,
                )
        # base model reference line for this category
        if base_record is not None:
            base_cm = base_cat_map.get(category)
            if base_cm is not None:
                base_val = base_cm.accuracy.get(match_type, MetricWithCI()).value
                _draw_base_hline(ax, base_val, color)
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Accuracy")
    match_label = MATCH_TYPE_LABELS[match_type]
    ax.set_title(f"{category_prefix.rstrip('/')} — {match_label} Accuracy")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8, loc="best")
    if warnings:
        print("Sample count drift detected (marked with *):")
        for warning in warnings:
            print(warning)
    return fig


def plot_token_usage_trends(
    records: list[CheckpointRecord],
    base_record: CheckpointRecord | None = None,
    metric_key: str = "attempt_token_usage/completion_tokens_mean",
    figsize: tuple[int, int] = FIGSIZE_SINGLE,
    std_alpha: float = CI_BAND_ALPHA,
) -> matplotlib.figure.Figure:
    """Plot a single token usage metric over training steps with ±1 std band."""
    steps = [r.step for r in records]
    tab10_colors = plt.cm.tab10.colors  # type: ignore[reportAttributeAccessIssue]
    fig, ax = plt.subplots(figsize=figsize)
    values = []
    std_values = []
    std_key = metric_key.replace("_mean", "_std") if metric_key.endswith("_mean") else None
    for record in records:
        val = record.eval_result.metrics.get(metric_key)
        values.append(float(val) if val is not None else np.nan)
        if std_key is not None:
            std_val = record.eval_result.metrics.get(std_key)
            std_values.append(float(std_val) if std_val is not None else np.nan)
    # clean up label: strip prefix and _mean suffix
    label = metric_key
    if "/" in label:
        label = label.split("/", 1)[1]
    label = label.removesuffix("_mean")
    color = tab10_colors[0]
    ax.plot(steps, values, "o-", color=color, label=label, linewidth=2, markersize=5)
    if std_key is not None and std_values and not all(np.isnan(std_values)):
        mean_arr = np.array(values)
        std_arr = np.array(std_values)
        ax.fill_between(steps, mean_arr - std_arr, mean_arr + std_arr, color=color, alpha=std_alpha)
    if base_record is not None:
        base_val = base_record.eval_result.metrics.get(metric_key)
        if base_val is not None:
            _draw_base_hline(ax, float(base_val), color, label=f"{label} (base)")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Tokens")
    ax.set_title(f"Token Usage over Training ({TARGET_EVAL_SUBSET_NAME} set, band = ±1 std)")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    return fig


def plot_pass_at_k_trends(
    records: list[CheckpointRecord],
    base_record: CheckpointRecord | None = None,
    figsize_per_subplot: tuple[int, int] = (6, 5),
    show_ci: bool = SHOW_CI_BANDS,
    ci_alpha: float = CI_BAND_ALPHA,
) -> matplotlib.figure.Figure:
    """Plot Pass@K metrics over training steps, one subplot per K value."""
    # collect all k values and match types present
    all_k_values = sorted(records[0].summary.run_info.pass_at_k.keys())
    if not all_k_values:
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.text(0.5, 0.5, "No Pass@K data available", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return fig
    num_subplots = len(all_k_values)
    fig, axes = plt.subplots(
        1,
        num_subplots,
        figsize=(figsize_per_subplot[0] * num_subplots, figsize_per_subplot[1]),
        squeeze=False,
    )
    steps = [r.step for r in records]
    tab10_colors = plt.cm.tab10.colors  # type: ignore[reportAttributeAccessIssue]
    for subplot_idx, k_val in enumerate(all_k_values):
        ax = axes[0, subplot_idx]
        # discover match types present for this K
        present_match_types = set()
        for record in records:
            k_data = record.summary.run_info.pass_at_k.get(k_val, {})
            present_match_types.update(k_data.keys())
        for mt_idx, match_type in enumerate(sorted(present_match_types)):
            y_vals = []
            ci_lo_vals = []
            ci_hi_vals = []
            for record in records:
                k_data = record.summary.run_info.pass_at_k.get(k_val, {})
                metric_ci = k_data.get(match_type, MetricWithCI())
                y_vals.append(metric_ci.value if metric_ci.value is not None else np.nan)
                ci_lo_vals.append(metric_ci.ci_lower if metric_ci.ci_lower is not None else np.nan)
                ci_hi_vals.append(metric_ci.ci_upper if metric_ci.ci_upper is not None else np.nan)
            color = tab10_colors[mt_idx % len(tab10_colors)]
            label = MATCH_TYPE_LABELS.get(match_type, match_type)  # type: ignore[arg-type]
            ax.plot(steps, y_vals, "o-", color=color, label=label, linewidth=2, markersize=5)
            if show_ci and not all(np.isnan(ci_lo_vals)) and not all(np.isnan(ci_hi_vals)):
                ax.fill_between(steps, ci_lo_vals, ci_hi_vals, color=color, alpha=ci_alpha)
            if base_record is not None:
                base_k_data = base_record.summary.run_info.pass_at_k.get(k_val, {})
                base_metric = base_k_data.get(match_type, MetricWithCI())
                _draw_base_hline(ax, base_metric.value, color, label=f"{label} (base)")
        ax.set_xlabel("Training Step")
        ax.set_ylabel("Pass@K")
        ax.set_title(f"Pass@{k_val}")
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", alpha=0.3)
        ax.legend()
    return fig

In [ ]:
# --- efficiency plotting helpers (accuracy vs generation tokens) ---

# the metric key used for the "generation cost" axis: reasoning tokens if available,
# otherwise completion tokens (covers both reasoning and non-reasoning models)
_GENERATION_TOKEN_CANDIDATES: list[str] = [
    "attempt_token_usage/reasoning_tokens_mean",
    "attempt_token_usage/completion_tokens_mean",
]


def _resolve_generation_token_key(
    record: CheckpointRecord,
    category: str | None = None,
) -> str | None:
    """Find the first non-NaN generation token metric key for a record."""
    prefix = f"{category}/" if category is not None else ""
    for candidate in _GENERATION_TOKEN_CANDIDATES:
        key = f"{prefix}{candidate}"
        val = record.eval_result.metrics.get(key)
        if val is not None and not np.isnan(float(val)):
            return key
    return None


def _get_generation_tokens_mean(
    record: CheckpointRecord,
    category: str | None = None,
    resolved_key: str | None = None,
) -> float:
    """Extract mean generation tokens (reasoning or completion) from metrics."""
    if resolved_key is not None:
        val = record.eval_result.metrics.get(resolved_key)
        return float(val) if val is not None else np.nan
    key = _resolve_generation_token_key(record, category)
    if key is None:
        return np.nan
    val = record.eval_result.metrics.get(key)
    return float(val) if val is not None else np.nan


def _resolve_generation_label(
    records: list[CheckpointRecord],
) -> str:
    """Return a human-readable label for the generation token axis."""
    key = _resolve_generation_token_key(records[0])
    if key is not None and "reasoning" in key:
        return "Reasoning Tokens"
    return "Completion Tokens"


def _annotate_trajectory_steps(
    ax: matplotlib.axes.Axes,
    x_vals: list[float],
    y_vals: list[float],
    steps: list[int],
    max_labels: int = 6,
    fontsize: int = 7,
) -> None:
    """Annotate a subset of trajectory points with step numbers."""
    num_points = len(steps)
    if num_points == 0:
        return
    indices = {0, num_points - 1}
    if num_points > 2:
        stride = max(1, (num_points - 1) // (max_labels - 1))
        for idx in range(stride, num_points - 1, stride):
            indices.add(idx)
    for idx in sorted(indices):
        if np.isnan(x_vals[idx]) or np.isnan(y_vals[idx]):
            continue
        ax.annotate(
            str(steps[idx]),
            (x_vals[idx], y_vals[idx]),
            textcoords="offset points",
            xytext=(4, 4),
            fontsize=fontsize,
            alpha=0.7,
        )


def _safe_efficiency_ratio(
    accuracy: np.ndarray,
    tokens: np.ndarray,
) -> np.ndarray:
    """Compute accuracy per 1K generation tokens, returning NaN where tokens <= 0."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(tokens > 0, accuracy / (tokens / 1000.0), np.nan)


def _draw_base_marker_on_trajectory(
    ax: matplotlib.axes.Axes,
    base_record: CheckpointRecord,
    match_type: pyine.evals.code_exec.utils.MatchType,
    color: typing.Any,
    category: str | None = None,
    label: str | None = "base",
) -> None:
    """Draw a distinct star marker for the base model on an efficiency trajectory plot."""
    if category is None:
        base_metric = base_record.summary.run_info.accuracy.get(match_type)
        base_acc = base_metric.value if base_metric is not None else None
    else:
        base_cat_map = {cm.category: cm for cm in base_record.summary.category_metrics}
        base_cm = base_cat_map.get(category)
        if base_cm is None:
            return
        base_acc = base_cm.accuracy.get(match_type, MetricWithCI()).value
    if base_acc is None or np.isnan(base_acc):
        return
    resolved_key = _resolve_generation_token_key(base_record, category)
    base_tokens = _get_generation_tokens_mean(base_record, category, resolved_key=resolved_key)
    if np.isnan(base_tokens):
        return
    ax.plot(
        base_tokens,
        base_acc,
        marker="*",
        color=color,
        markersize=14,
        markeredgecolor="black",
        markeredgewidth=0.8,
        zorder=5,
        label=label,
    )


def plot_efficiency_trajectory(
    records: list[CheckpointRecord],
    match_type: pyine.evals.code_exec.utils.MatchType = "hard",
    category_prefix: str | None = None,
    ax: matplotlib.axes.Axes | None = None,
    base_record: CheckpointRecord | None = None,
    figsize: tuple[int, int] = FIGSIZE_SINGLE,
    show_ci: bool = SHOW_CI_BANDS,
    annotate_steps: bool = True,
) -> matplotlib.figure.Figure:
    """Plot accuracy vs mean generation tokens, connected by checkpoint order.

    Each point is a checkpoint. Line direction shows training progression.
    Overall (category_prefix=None): one line for the given match type.
    Per-category: one line per category.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = typing.cast("matplotlib.figure.Figure", ax.get_figure())
    tab10_colors = plt.cm.tab10.colors  # type: ignore[reportAttributeAccessIssue]
    steps = [r.step for r in records]
    token_label = _resolve_generation_label(records)
    if category_prefix is None:
        # resolve key once from first record, reuse for consistency
        resolved_key = _resolve_generation_token_key(records[0])
        x_vals = [_get_generation_tokens_mean(r, resolved_key=resolved_key) for r in records]
        acc_data = [r.summary.run_info.accuracy.get(match_type) for r in records]
        y_vals = [v.value if v is not None else np.nan for v in acc_data]
        mt_idx = list(MATCH_TYPES).index(match_type) if match_type in MATCH_TYPES else 0
        color = tab10_colors[mt_idx % len(tab10_colors)]
        ax.plot(x_vals, y_vals, "o-", color=color, label=MATCH_TYPE_LABELS[match_type], linewidth=2, markersize=5)
        if show_ci:
            y_arr = np.array(y_vals)
            ci_lo = np.array([v.ci_lower if v is not None and v.ci_lower is not None else np.nan for v in acc_data])
            ci_hi = np.array([v.ci_upper if v is not None and v.ci_upper is not None else np.nan for v in acc_data])
            valid = ~(np.isnan(y_arr) | np.isnan(ci_lo) | np.isnan(ci_hi))
            if valid.any():
                ax.errorbar(
                    np.array(x_vals)[valid],
                    y_arr[valid],
                    yerr=[y_arr[valid] - ci_lo[valid], ci_hi[valid] - y_arr[valid]],
                    fmt="none",
                    color=color,
                    alpha=0.4,
                    capsize=2,
                    capthick=0.8,
                    linewidth=0.8,
                )
        if annotate_steps:
            _annotate_trajectory_steps(ax, x_vals, y_vals, steps)
        if base_record is not None:
            _draw_base_marker_on_trajectory(ax, base_record, match_type, color)
    else:
        all_categories = sorted(
            {
                cm.category
                for r in records
                for cm in r.summary.category_metrics
                if cm.category.startswith(category_prefix)
            }
        )
        cat_lookup: dict[int, dict[str, pyine.evals.code_exec.analysis.CategoryMetrics]] = {}
        for record in records:
            cat_lookup[record.step] = {
                cm.category: cm for cm in record.summary.category_metrics if cm.category.startswith(category_prefix)
            }
        for cat_idx, category in enumerate(all_categories):
            short_label = category.removeprefix(category_prefix)
            color = tab10_colors[cat_idx % len(tab10_colors)]
            resolved_key = _resolve_generation_token_key(records[0], category)
            x_vals = [_get_generation_tokens_mean(r, category, resolved_key=resolved_key) for r in records]
            y_vals_list: list[float] = []
            ci_lo_list: list[float] = []
            ci_hi_list: list[float] = []
            for record in records:
                cm = cat_lookup[record.step].get(category)
                if cm is None:
                    y_vals_list.append(np.nan)
                    ci_lo_list.append(np.nan)
                    ci_hi_list.append(np.nan)
                else:
                    metric_ci = cm.accuracy.get(match_type, MetricWithCI())
                    y_vals_list.append(metric_ci.value if metric_ci.value is not None else np.nan)
                    ci_lo_list.append(metric_ci.ci_lower if metric_ci.ci_lower is not None else np.nan)
                    ci_hi_list.append(metric_ci.ci_upper if metric_ci.ci_upper is not None else np.nan)
            ax.plot(x_vals, y_vals_list, "o-", color=color, label=short_label, linewidth=1.5, markersize=4)
            if show_ci:
                y_arr = np.array(y_vals_list)
                ci_lo_arr = np.array(ci_lo_list)
                ci_hi_arr = np.array(ci_hi_list)
                valid = ~(np.isnan(y_arr) | np.isnan(ci_lo_arr) | np.isnan(ci_hi_arr))
                if valid.any():
                    ax.errorbar(
                        np.array(x_vals)[valid],
                        y_arr[valid],
                        yerr=[y_arr[valid] - ci_lo_arr[valid], ci_hi_arr[valid] - y_arr[valid]],
                        fmt="none",
                        color=color,
                        alpha=0.35,
                        capsize=2,
                        capthick=0.7,
                        linewidth=0.7,
                    )
            if base_record is not None:
                _draw_base_marker_on_trajectory(
                    ax,
                    base_record,
                    match_type,
                    color,
                    category=category,
                    label=f"{short_label} (base)" if cat_idx == 0 else None,
                )
        # annotate steps on first category only to avoid clutter
        if annotate_steps and all_categories:
            first_cat = all_categories[0]
            resolved_key = _resolve_generation_token_key(records[0], first_cat)
            x_first = [_get_generation_tokens_mean(r, first_cat, resolved_key=resolved_key) for r in records]
            y_first: list[float] = []
            for record in records:
                cm = cat_lookup[record.step].get(first_cat)
                if cm is None:
                    y_first.append(np.nan)
                else:
                    m = cm.accuracy.get(match_type, MetricWithCI())
                    y_first.append(m.value if m.value is not None else np.nan)
            _annotate_trajectory_steps(ax, x_first, y_first, steps, fontsize=6)
    ax.set_xlabel(f"Mean {token_label}")
    ax.set_ylabel("Accuracy")
    match_label = MATCH_TYPE_LABELS[match_type]
    prefix_label = f"{category_prefix.rstrip('/')} — " if category_prefix else "Overall "
    ax.set_title(f"{prefix_label}{match_label} Accuracy vs {token_label}")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc="best")
    return fig


def plot_efficiency_ratio(
    records: list[CheckpointRecord],
    match_type: pyine.evals.code_exec.utils.MatchType = "hard",
    category_prefix: str | None = None,
    ax: matplotlib.axes.Axes | None = None,
    base_record: CheckpointRecord | None = None,
    figsize: tuple[int, int] = FIGSIZE_SINGLE,
    show_ci: bool = SHOW_CI_BANDS,
) -> matplotlib.figure.Figure:
    """Plot accuracy per 1K generation tokens over training steps.

    Higher = more performance per unit of generation. Shows whether training
    improves efficiency (accuracy gains outpace token usage) or just spends more tokens.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = typing.cast("matplotlib.figure.Figure", ax.get_figure())
    tab10_colors = plt.cm.tab10.colors  # type: ignore[reportAttributeAccessIssue]
    steps = np.array([r.step for r in records])
    token_label = _resolve_generation_label(records)
    if category_prefix is None:
        resolved_key = _resolve_generation_token_key(records[0])
        tokens = np.array([_get_generation_tokens_mean(r, resolved_key=resolved_key) for r in records])
        acc_data = [r.summary.run_info.accuracy.get(match_type) for r in records]
        acc = np.array([v.value if v is not None else np.nan for v in acc_data])
        ratio = _safe_efficiency_ratio(acc, tokens)
        mt_idx = list(MATCH_TYPES).index(match_type) if match_type in MATCH_TYPES else 0
        color = tab10_colors[mt_idx % len(tab10_colors)]
        ax.plot(steps, ratio, "o-", color=color, label=MATCH_TYPE_LABELS[match_type], linewidth=2, markersize=5)
        if show_ci:
            ci_lo = np.array([v.ci_lower if v is not None and v.ci_lower is not None else np.nan for v in acc_data])
            ci_hi = np.array([v.ci_upper if v is not None and v.ci_upper is not None else np.nan for v in acc_data])
            ratio_lo = _safe_efficiency_ratio(ci_lo, tokens)
            ratio_hi = _safe_efficiency_ratio(ci_hi, tokens)
            valid = ~(np.isnan(ratio) | np.isnan(ratio_lo) | np.isnan(ratio_hi))
            if valid.any():
                ax.fill_between(steps[valid], ratio_lo[valid], ratio_hi[valid], color=color, alpha=CI_BAND_ALPHA)
        if base_record is not None:
            base_resolved = _resolve_generation_token_key(base_record)
            base_tokens = _get_generation_tokens_mean(base_record, resolved_key=base_resolved)
            base_metric = base_record.summary.run_info.accuracy.get(match_type)
            base_acc = base_metric.value if base_metric is not None else np.nan
            base_ratio = _safe_efficiency_ratio(np.array([base_acc]), np.array([base_tokens]))
            _draw_base_hline(ax, float(base_ratio[0]), color, label=f"{MATCH_TYPE_LABELS[match_type]} (base)")
    else:
        all_categories = sorted(
            {
                cm.category
                for r in records
                for cm in r.summary.category_metrics
                if cm.category.startswith(category_prefix)
            }
        )
        cat_lookup: dict[int, dict[str, pyine.evals.code_exec.analysis.CategoryMetrics]] = {}
        for record in records:
            cat_lookup[record.step] = {
                cm.category: cm for cm in record.summary.category_metrics if cm.category.startswith(category_prefix)
            }
        # build base model category lookup
        base_cat_map: dict[str, pyine.evals.code_exec.analysis.CategoryMetrics] = {}
        if base_record is not None:
            base_cat_map = {
                cm.category: cm
                for cm in base_record.summary.category_metrics
                if cm.category.startswith(category_prefix)
            }
        jitter = _compute_category_jitter_offsets(list(steps), len(all_categories))
        for cat_idx, category in enumerate(all_categories):
            short_label = category.removeprefix(category_prefix)
            color = tab10_colors[cat_idx % len(tab10_colors)]
            resolved_key = _resolve_generation_token_key(records[0], category)
            tokens = np.array([_get_generation_tokens_mean(r, category, resolved_key=resolved_key) for r in records])
            acc_vals: list[float] = []
            ci_lo_vals: list[float] = []
            ci_hi_vals: list[float] = []
            for record in records:
                cm = cat_lookup[record.step].get(category)
                if cm is None:
                    acc_vals.append(np.nan)
                    ci_lo_vals.append(np.nan)
                    ci_hi_vals.append(np.nan)
                else:
                    metric_ci = cm.accuracy.get(match_type, MetricWithCI())
                    acc_vals.append(metric_ci.value if metric_ci.value is not None else np.nan)
                    ci_lo_vals.append(metric_ci.ci_lower if metric_ci.ci_lower is not None else np.nan)
                    ci_hi_vals.append(metric_ci.ci_upper if metric_ci.ci_upper is not None else np.nan)
            acc = np.array(acc_vals)
            ratio = _safe_efficiency_ratio(acc, tokens)
            x_jittered = steps + jitter[cat_idx]
            ax.plot(x_jittered, ratio, "o-", color=color, label=short_label, linewidth=1.5, markersize=4)
            if show_ci:
                ci_lo_arr = np.array(ci_lo_vals)
                ci_hi_arr = np.array(ci_hi_vals)
                ratio_lo = _safe_efficiency_ratio(ci_lo_arr, tokens)
                ratio_hi = _safe_efficiency_ratio(ci_hi_arr, tokens)
                valid = ~(np.isnan(ratio) | np.isnan(ratio_lo) | np.isnan(ratio_hi))
                if valid.any():
                    ax.errorbar(
                        x_jittered[valid],
                        ratio[valid],
                        yerr=[ratio[valid] - ratio_lo[valid], ratio_hi[valid] - ratio[valid]],
                        fmt="none",
                        color=color,
                        alpha=0.35,
                        capsize=2,
                        capthick=0.7,
                        linewidth=0.7,
                    )
            # base model reference line for this category
            if base_record is not None:
                base_cm = base_cat_map.get(category)
                if base_cm is not None:
                    base_acc = base_cm.accuracy.get(match_type, MetricWithCI()).value
                    base_resolved = _resolve_generation_token_key(base_record, category)
                    base_tokens = _get_generation_tokens_mean(base_record, category, resolved_key=base_resolved)
                    if base_acc is not None and not np.isnan(base_tokens):
                        base_ratio = _safe_efficiency_ratio(np.array([base_acc]), np.array([base_tokens]))
                        _draw_base_hline(ax, float(base_ratio[0]), color)
    ax.set_xlabel("Training Step")
    ax.set_ylabel(f"Accuracy per 1K {token_label}")
    match_label = MATCH_TYPE_LABELS[match_type]
    prefix_label = f"{category_prefix.rstrip('/')} — " if category_prefix else "Overall "
    ax.set_title(f"{prefix_label}{match_label} Efficiency Ratio")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8, loc="best")
    return fig

In [ ]:
# --- overall training curves ---
fig = plot_training_curves(records, base_record=base_record)
plt.tight_layout()
plt.show()

In [ ]:
# --- code type training curves ---
# determine which match types have non-None accuracy data across any checkpoint
active_match_types: list[pyine.evals.code_exec.utils.MatchType] = []
for match_type in MATCH_TYPES:
    has_data = any(
        any(
            cm.accuracy.get(match_type, MetricWithCI()).value is not None
            for cm in record.summary.category_metrics
            if cm.category.startswith("code_type/")
        )
        for record in records
    )
    if has_data:
        active_match_types.append(match_type)

if active_match_types:
    num_plots = len(active_match_types)
    fig, axes = plt.subplots(1, num_plots, figsize=(FIGSIZE_MULTI[0], FIGSIZE_MULTI[1]), squeeze=False)
    for plot_idx, match_type in enumerate(active_match_types):
        plot_category_training_curves(
            records,
            "code_type/",
            match_type,
            ax=axes[0, plot_idx],
            base_record=base_record,
        )
    plt.tight_layout()
    plt.show()
    # show per-category sample count table across checkpoints
    code_type_categories = sorted(
        {cm.category for r in records for cm in r.summary.category_metrics if cm.category.startswith("code_type/")}
    )
    count_rows = []
    for category in code_type_categories:
        row: dict[str, typing.Any] = {"category": category.removeprefix("code_type/")}
        for record in records:
            cat_map = {cm.category: cm for cm in record.summary.category_metrics}
            cm = cat_map.get(category)
            row[f"step_{record.step}"] = cm.sample_count if cm is not None else 0
        count_rows.append(row)
    count_df = pd.DataFrame(count_rows)
    step_cols = [col for col in count_df.columns if col.startswith("step_")]
    has_any_drift = any(count_df[step_cols].nunique(axis=1) > 1)
    if has_any_drift:
        print("WARNING: sample counts differ across checkpoints for some code types (see table).")
    display(count_df)  # type: ignore[name-defined]  # noqa: F821
else:
    print("No code_type category data found")

In [ ]:
# --- predict type training curves ---
active_predict_match_types: list[pyine.evals.code_exec.utils.MatchType] = []
for match_type in MATCH_TYPES:
    has_data = any(
        any(
            cm.accuracy.get(match_type, MetricWithCI()).value is not None
            for cm in record.summary.category_metrics
            if cm.category.startswith("predict_type/")
        )
        for record in records
    )
    if has_data:
        active_predict_match_types.append(match_type)

if active_predict_match_types:
    num_plots = len(active_predict_match_types)
    fig, axes = plt.subplots(1, num_plots, figsize=(FIGSIZE_MULTI[0], FIGSIZE_MULTI[1]), squeeze=False)
    for plot_idx, match_type in enumerate(active_predict_match_types):
        plot_category_training_curves(
            records,
            "predict_type/",
            match_type,
            ax=axes[0, plot_idx],
            base_record=base_record,
        )
    plt.tight_layout()
    plt.show()
    # show per-category sample count table across checkpoints
    predict_type_categories = sorted(
        {cm.category for r in records for cm in r.summary.category_metrics if cm.category.startswith("predict_type/")}
    )
    count_rows = []
    for category in predict_type_categories:
        row: dict[str, typing.Any] = {"category": category.removeprefix("predict_type/")}
        for record in records:
            cat_map = {cm.category: cm for cm in record.summary.category_metrics}
            cm = cat_map.get(category)
            row[f"step_{record.step}"] = cm.sample_count if cm is not None else 0
        count_rows.append(row)
    count_df = pd.DataFrame(count_rows)
    step_cols = [col for col in count_df.columns if col.startswith("step_")]
    has_any_drift = any(count_df[step_cols].nunique(axis=1) > 1)
    if has_any_drift:
        print("WARNING: sample counts differ across checkpoints for some predict types (see table).")
    display(count_df)  # type: ignore[name-defined]  # noqa: F821
else:
    print("No predict_type category data found")

In [ ]:
# --- token usage trends ---
fig = plot_token_usage_trends(records, base_record=base_record)
plt.tight_layout()
plt.show()

In [ ]:
# --- Pass@K trends (conditional) ---
has_pass_at_k = bool(records[0].summary.run_info.pass_at_k)
if has_pass_at_k:
    fig = plot_pass_at_k_trends(records, base_record=base_record)
    plt.tight_layout()
    plt.show()
else:
    print("No Pass@K data available (single-attempt evaluation or K=1 only).")

In [ ]:
# --- overall accuracy vs generation tokens ---
active_overall_mt: list[pyine.evals.code_exec.utils.MatchType] = [
    mt for mt in MATCH_TYPES if records[0].summary.run_info.accuracy.get(mt) is not None
]
generation_available = _resolve_generation_token_key(records[0]) is not None

if active_overall_mt and generation_available:
    fig, (ax_traj, ax_ratio) = plt.subplots(1, 2, figsize=(FIGSIZE_MULTI[0], FIGSIZE_MULTI[1]))
    token_label = _resolve_generation_label(records)
    for mt_idx, match_type in enumerate(active_overall_mt):
        plot_efficiency_trajectory(
            records,
            match_type=match_type,
            ax=ax_traj,
            base_record=base_record,
            annotate_steps=(mt_idx == 0),  # annotate only on first line
        )
        plot_efficiency_ratio(records, match_type=match_type, ax=ax_ratio, base_record=base_record)
    ax_traj.set_title(f"Accuracy vs {token_label} ({TARGET_EVAL_SUBSET_NAME} set)")
    ax_ratio.set_title(f"Efficiency Ratio ({TARGET_EVAL_SUBSET_NAME} set)")
    plt.tight_layout()
    plt.show()
else:
    print("No generation token data available for efficiency plots")

In [ ]:
# --- code type accuracy vs generation tokens ---
active_ct_eff_mt: list[pyine.evals.code_exec.utils.MatchType] = []
for match_type in MATCH_TYPES:
    has_data = any(
        any(
            cm.accuracy.get(match_type, MetricWithCI()).value is not None
            for cm in record.summary.category_metrics
            if cm.category.startswith("code_type/")
        )
        for record in records
    )
    if has_data:
        active_ct_eff_mt.append(match_type)

if active_ct_eff_mt and generation_available:
    num_mt = len(active_ct_eff_mt)
    fig, axes = plt.subplots(2, num_mt, figsize=(7 * num_mt, 10), squeeze=False)
    for mt_idx, match_type in enumerate(active_ct_eff_mt):
        plot_efficiency_trajectory(
            records,
            match_type=match_type,
            category_prefix="code_type/",
            ax=axes[0, mt_idx],
            base_record=base_record,
        )
        plot_efficiency_ratio(
            records,
            match_type=match_type,
            category_prefix="code_type/",
            ax=axes[1, mt_idx],
            base_record=base_record,
        )
    plt.tight_layout()
    plt.show()
else:
    print("No code_type category data or generation tokens available for efficiency plots")

In [ ]:
# --- predict type accuracy vs generation tokens ---
active_pt_eff_mt: list[pyine.evals.code_exec.utils.MatchType] = []
for match_type in MATCH_TYPES:
    has_data = any(
        any(
            cm.accuracy.get(match_type, MetricWithCI()).value is not None
            for cm in record.summary.category_metrics
            if cm.category.startswith("predict_type/")
        )
        for record in records
    )
    if has_data:
        active_pt_eff_mt.append(match_type)

if active_pt_eff_mt and generation_available:
    num_mt = len(active_pt_eff_mt)
    fig, axes = plt.subplots(2, num_mt, figsize=(7 * num_mt, 10), squeeze=False)
    for mt_idx, match_type in enumerate(active_pt_eff_mt):
        plot_efficiency_trajectory(
            records,
            match_type=match_type,
            category_prefix="predict_type/",
            ax=axes[0, mt_idx],
            base_record=base_record,
        )
        plot_efficiency_ratio(
            records,
            match_type=match_type,
            category_prefix="predict_type/",
            ax=axes[1, mt_idx],
            base_record=base_record,
        )
    plt.tight_layout()
    plt.show()
else:
    print("No predict_type category data or generation tokens available for efficiency plots")

In [ ]:
# --- consistency & diagnostics summary ---
diag_rows = []
all_records = ([base_record] if base_record is not None else []) + records
for record in all_records:
    num_samples = record.eval_result.num_samples
    num_attempts = record.eval_result.num_attempts
    attempts_per_sample = num_attempts // num_samples if num_samples > 0 else 0
    pass_k_values = sorted(record.summary.run_info.pass_at_k.keys()) if record.summary.run_info.pass_at_k else []
    diag_rows.append(
        {
            "checkpoint_step": "base" if record.is_base_model else record.step,
            "sample_count": num_samples,
            "attempt_count": num_attempts,
            "attempts_per_sample": attempts_per_sample,
            "pass_at_k_values": str(pass_k_values) if pass_k_values else "none",
            "lmdb_path": str(record.lmdb_path),
        }
    )
diag_df = pd.DataFrame(diag_rows)
print("Diagnostics summary (all values should be consistent across checkpoints):")
diag_df  # noqa: B018

In [ ]:
# --- paper-quality 3-panel figure ---

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 14,
        "axes.titlesize": 15,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

fig, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=(11, 3))
tab10_colors = plt.cm.tab10.colors  # type: ignore[reportAttributeAccessIssue]
# panel (a) uses: tab10[0]=blue (hinted), tab10[1]=orange (misleading), tab10[2]=green (original)

# (a) code_type soft-match accuracy (reuse existing helper with base model)
plot_category_training_curves(
    records,
    "code_type/",
    "soft",
    ax=ax_a,
    base_record=base_record,
    ci_capsize=4,
    ci_capthick=1.2,
    ci_linewidth=1.2,
    ci_alpha=0.7,
)
ax_a.set_title("(a) Soft-match accuracy")

# (b) token usage
steps = [r.step for r in records]
token_metric_key = "attempt_token_usage/completion_tokens_mean"  # noqa: S105
token_std_key = "attempt_token_usage/completion_tokens_std"  # noqa: S105
token_vals = []
token_std_vals = []
for record in records:
    val = record.eval_result.metrics.get(token_metric_key)
    token_vals.append(float(val) if val is not None else np.nan)
    std_val = record.eval_result.metrics.get(token_std_key)
    token_std_vals.append(float(std_val) if std_val is not None else np.nan)
color_b = tab10_colors[4]  # purple, distinct from (a)'s blue/orange/green
ax_b.plot(steps, token_vals, "o-", color=color_b, linewidth=2, markersize=5)
if not all(np.isnan(token_std_vals)):
    mean_arr = np.array(token_vals)
    std_arr = np.array(token_std_vals)
    ax_b.fill_between(steps, mean_arr - std_arr, mean_arr + std_arr, color=color_b, alpha=CI_BAND_ALPHA)
ax_b.set_xlabel("Training Step")
ax_b.set_ylabel("Tokens")
ax_b.set_title("(b) Response length")
ax_b.set_ylim(bottom=0)
ax_b.grid(axis="y", alpha=0.3)

# (c) efficiency ratio, original-task soft match only
_original_category = "code_type/original"
resolved_key = _resolve_generation_token_key(records[0], _original_category)
tokens_arr = np.array([_get_generation_tokens_mean(r, _original_category, resolved_key=resolved_key) for r in records])
acc_data: list[MetricWithCI | None] = []
for record in records:
    cat_map = {cm.category: cm for cm in record.summary.category_metrics}
    orig_cm = cat_map.get(_original_category)
    if orig_cm is not None:
        acc_data.append(orig_cm.accuracy.get("soft"))
    else:
        acc_data.append(None)
acc_arr = np.array([v.value if v is not None else np.nan for v in acc_data])
ratio_arr = _safe_efficiency_ratio(acc_arr, tokens_arr)
color_c = tab10_colors[2]  # green — matches "original" in panel (a)
ax_c.plot(steps, ratio_arr, "o-", color=color_c, linewidth=2, markersize=5)
if SHOW_CI_BANDS:
    ci_lo = np.array([v.ci_lower if v is not None and v.ci_lower is not None else np.nan for v in acc_data])
    ci_hi = np.array([v.ci_upper if v is not None and v.ci_upper is not None else np.nan for v in acc_data])
    ratio_lo = _safe_efficiency_ratio(ci_lo, tokens_arr)
    ratio_hi = _safe_efficiency_ratio(ci_hi, tokens_arr)
    valid = ~(np.isnan(ratio_arr) | np.isnan(ratio_lo) | np.isnan(ratio_hi))
    if valid.any():
        ax_c.fill_between(
            np.array(steps)[valid],
            ratio_lo[valid],
            ratio_hi[valid],
            color=color_c,
            alpha=CI_BAND_ALPHA,
        )
ax_c.axvline(600, color="grey", linestyle="--", linewidth=1.2, alpha=0.7, label="selected (step 600)")
ax_c.set_xlabel("Training Step")
ax_c.set_ylabel("Accuracy / 1K output tokens")
ax_c.set_title("(c) Efficiency ratio (original)")
ax_c.grid(axis="y", alpha=0.3)
ax_c.legend(loc="lower left")

plt.tight_layout()

notebook_artifacts_path = pyine.utils.filesystem.get_logs_root_path() / "paper_figures"
notebook_artifacts_path.mkdir(parents=True, exist_ok=True)
fig.savefig(notebook_artifacts_path / "checkpoint_progression.pdf", bbox_inches="tight")
fig.savefig(notebook_artifacts_path / "checkpoint_progression.png", bbox_inches="tight")
plt.show()
print(f"saved to {notebook_artifacts_path}")